Model trained on all data from 2005 - 2022 and then tested on 2023 March Madness Tournament

In [2]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score

In [7]:
## Data preparation

# Load Data
df = pd.read_csv("InputData.csv")

# Filtering to only 2023 games
df['Date'] = pd.to_datetime(df['Date'])
test_data = df[(df['Date'] >= pd.Timestamp('2023-03-16')) & (df['is_tournament_game'] == True)]
train_data = df[df['Date'] <= pd.Timestamp('2023-03-15')]

# Keeping metadata columns
columns = ['Game ID', 'Team 1', 'Team 2', 'Date', 'Site', 'Outcome', 'is_tournament_game']
metadata_train = train_data[columns].copy()
metadata_test = test_data[columns].copy()

# Prepare train data
X_train = train_data.drop(columns=columns)
y_train = train_data['Outcome']

# Prepare test data
X_test = test_data.drop(columns=columns)
y_test = test_data['Outcome']

# Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
## Train the model

# Create and train Logistic Regression Model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

In [9]:
## Evaluate the model

print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Add predictions to metadata for review
metadata_test['Predicted_Outcome'] = y_pred
metadata_test['Actual_Outcome'] = y_test.values
metadata_test['Prediction_Probability'] = y_pred_proba.max(axis=1)

# Export predictions to CSV
metadata_test.to_csv('predictions.csv', index=False)
print("\nPredictions exported to 'predictions.csv'")

Model Accuracy: 0.6190476190476191

Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.69      0.60        26
           1       0.72      0.57      0.64        37

    accuracy                           0.62        63
   macro avg       0.63      0.63      0.62        63
weighted avg       0.64      0.62      0.62        63


Confusion Matrix:
[[18  8]
 [16 21]]

Predictions exported to 'predictions.csv'


In [ ]:

# Define models to test
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

# Store results
results = []

# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    results.append({
        'Model': name,
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Test Precision': precision_score(y_test, y_test_pred),
        'Test Recall': recall_score(y_test, y_test_pred),
        'Test F1': f1_score(y_test, y_test_pred)
    })

# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test Accuracy', ascending=False)

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(results_df.to_string(index=False))

# Export results
results_df.to_csv('model_comparison2023.csv', index=False)
print("\nResults exported to 'model_comparison2023.csv'")

# Get the best model
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"\nBest Model: {best_model_name}")
print(f"Test Accuracy: {results_df.iloc[0]['Test Accuracy']:.4f}")

# Make predictions with best model
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)

# Update metadata with best model predictions
metadata_test['Predicted_Outcome'] = y_pred_best
metadata_test['Actual_Outcome'] = y_test.values
metadata_test['Prediction_Probability'] = y_pred_proba_best.max(axis=1)
metadata_test['Model_Used'] = best_model_name

# Export predictions
metadata_test.to_csv('predictions_best_model2023.csv', index=False)
print(f"\nPredictions from best model exported to 'predictions_best_model2023.csv'")